In [1]:
import torch
import os

In [2]:
class Value:
    def __init__(self,data, _child=(), _op=''):
        self.data = data
        self._prev = set(_child)
        self._op = _op
        
    def __repr__(self):
        return f"Value(data = {self.data})"

    def __add__(self, other):
        out = Value(self.data + other.data, (self,other), '+')
        return out
    def __mul__(self,other):
        out = Value(self.data * other.data, (self,other), '*')
        return out

In [3]:
a = 2.0
print("a is at :", a)

a is at : 2.0


In [4]:
a = Value(2.0)
print("a after repr: ", a)

a after repr:  Value(data = 2.0)


In [5]:
a = Value(5.0)
b = Value(2.0)
print(f"a= {a}\nb= {b}")

a+b

a= Value(data = 5.0)
b= Value(data = 2.0)


Value(data = 7.0)

In [6]:
a * b

Value(data = 10.0)

In [7]:
a = Value(3.0)
b = Value(4.0)
c = Value(-2.0)

print(f"a= {a}\nb= {b}\nc = {c}")

d = a*b +c
d

a= Value(data = 3.0)
b= Value(data = 4.0)
c = Value(data = -2.0)


Value(data = 10.0)

In [8]:
d._prev

{Value(data = -2.0), Value(data = 12.0)}

In [9]:
d._op

'+'

In [10]:
import os
import shutil
import sys
from graphviz import Digraph

# Jupyter kernels often omit the conda env bin from PATH, so `dot` is missing
# even when Graphviz is installed. Put this interpreter's bin first.
_env_bin = os.path.dirname(sys.executable)
if _env_bin not in os.environ.get("PATH", "").split(os.pathsep):
    os.environ["PATH"] = _env_bin + os.pathsep + os.environ.get("PATH", "")

if shutil.which("dot") is None:
    raise RuntimeError(
        "Graphviz 'dot' not found. In the notebook kernel env run: "
        "conda install -y graphviz"
    )

In [13]:
def trace(root):
    #builds a set of all nodes and edges in the graph
    nodes, edges = set(), set()
    
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)
    
    build(root)
    return nodes, edges

def draw_dot(root):
    dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'}) # LR = left to right
    nodes, edges = trace(root)
    
    for n in nodes:
        uid = str(id(n))
        # for any node in the grapg create a rectangle structure
        dot.node(name = uid, label = "{ data %.4f }" % (n.data, ), shape='record')
        
        if n._op:
            # if this value is a result of an some operation, create a label for it and add it to the node
            dot.node(name = uid + n._op, label = n._op, shape='record')
            # and connect this node to it!!!
            dot.edge(uid + n._op, uid)
            
    for n1, n2 in edges:
        # connect n1 to the op node of n2
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)
    
    return dot

In [ ]:
draw_dot(d)